In [3]:
pip install opencv-contrib-python

In [2]:
import cv2
import numpy as np

# --- CONFIGURAÇÕES DE COR OTIMIZADAS ---

# AZUL (Fundo da caneta)
# Aumentei o H e o V mínimos para separar bem do Verde
AZUL_LOW = np.array([100, 110, 130]) 
AZUL_HIGH = np.array([115, 255, 255])

# VERDE (Meio da caneta)
# H vai até 98 (antes do azul), e o V máximo é 120 (abaixo do azul)
VERDE_LOW = np.array([75, 100, 10])
VERDE_HIGH = np.array([98, 255, 120])

FATOR_K = 3.6

def encontrar_centroide(mask):
    contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contornos:
        maior_contorno = max(contornos, key=cv2.contourArea)
        if cv2.contourArea(maior_contorno) > 100:
            M = cv2.moments(maior_contorno)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                return (cx, cy)
    return None

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret: break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    mask_azul = cv2.inRange(hsv, AZUL_LOW, AZUL_HIGH)
    mask_verde = cv2.inRange(hsv, VERDE_LOW, VERDE_HIGH)

    kernel = np.ones((5,5),np.uint8)
    mask_azul = cv2.morphologyEx(mask_azul, cv2.MORPH_OPEN, kernel)
    mask_verde = cv2.morphologyEx(mask_verde, cv2.MORPH_OPEN, kernel)

    centro_azul = encontrar_centroide(mask_azul)
    centro_verde = encontrar_centroide(mask_verde)

    if centro_azul and centro_verde:
        # Desenha os círculos de marcação
        cv2.circle(frame, centro_azul, 10, (255, 0, 0), -1)      # Azul
        cv2.circle(frame, centro_verde, 10, (0, 255, 0), -1)     # Verde

        # Vetor: Sai do Azul (fundo) apontando para o Verde (meio)
        vetor_x = centro_verde[0] - centro_azul[0]
        vetor_y = centro_verde[1] - centro_azul[1]

        # Estica o vetor a partir do Verde para achar a ponta
        ponta_x = int(centro_verde[0] + (vetor_x * FATOR_K))
        ponta_y = int(centro_verde[1] + (vetor_y * FATOR_K))
        ponto_ponta = (ponta_x, ponta_y)

        # Desenha a linha e o alvo na ponta
        cv2.line(frame, centro_azul, ponto_ponta, (255, 255, 255), 2)
        cv2.drawMarker(frame, ponto_ponta, (0, 0, 255), cv2.MARKER_CROSS, 20, 3)

    cv2.imshow('Rastreio de Caneta (Verde/Azul)', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [1]:
import cv2
import numpy as np

# --- CONFIGURAÇÕES DO CHARUCO (Seu PDF) ---
CANTOS_X = 11
CANTOS_Y = 8
TAMANHO_QUADRADO_M = 0.015  # 15 mm
TAMANHO_MARCADOR_M = 0.011  # 11 mm

# Configura o Dicionário e o Tabuleiro
dicionario = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
tabuleiro = cv2.aruco.CharucoBoard((CANTOS_X, CANTOS_Y), TAMANHO_QUADRADO_M, TAMANHO_MARCADOR_M, dicionario)

# Inicia o Detector moderno do OpenCV 4.11
charuco_detector = cv2.aruco.CharucoDetector(tabuleiro)

todos_cantos = []
todos_ids = []
tamanho_imagem = None

cap = cv2.VideoCapture(0)
fotos_tiradas = 0

print("=== CALIBRAÇÃO DE CÂMERA INICIADA ===")
print("- Pressione 'c' para CAPTURAR uma foto válida.")
print("- Pressione 'q' quando tiver cerca de 20 a 30 fotos para FINALIZAR.")

while True:
    ret, frame = cap.read()
    if not ret: break

    if tamanho_imagem is None:
        tamanho_imagem = (frame.shape[1], frame.shape[0])

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frame_desenho = frame.copy()

    # OpenCV 4.11+: O detector faz tudo em um passo só!
    charuco_cantos, charuco_ids, marcadores_cantos, marcadores_ids = charuco_detector.detectBoard(gray)

    # Se achou os códigos de barra (marcadores), desenha o contorno deles
    if marcadores_cantos is not None and len(marcadores_cantos) > 0:
        cv2.aruco.drawDetectedMarkers(frame_desenho, marcadores_cantos, marcadores_ids)

        # Se achou os cantos do xadrez com base nos códigos, desenha e libera a captura
        if charuco_cantos is not None and charuco_ids is not None and len(charuco_cantos) > 3:
            cv2.aruco.drawDetectedCornersCharuco(frame_desenho, charuco_cantos, charuco_ids, (0, 255, 0))
            
            cv2.putText(frame_desenho, "Pressione 'c' para capturar", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            tecla = cv2.waitKey(1) & 0xFF
            if tecla == ord('c'):
                todos_cantos.append(charuco_cantos)
                todos_ids.append(charuco_ids)
                fotos_tiradas += 1
                print(f"Foto {fotos_tiradas} capturada! Mude o ângulo do tabuleiro.")
    else:
        tecla = cv2.waitKey(1) & 0xFF

    cv2.putText(frame_desenho, f"Fotos: {fotos_tiradas}/20+", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.imshow("Calibracao CharUco", frame_desenho)

    if tecla == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# --- FASE 2: MATEMÁTICA DA CALIBRAÇÃO ---
if len(todos_cantos) > 0:
    print("\nCalculando matriz da lente... (isso pode levar alguns segundos)")
    
    # Função de calibração também otimizada
    erro, matriz_camera, coef_distorcao, rvecs, tvecs = cv2.aruco.calibrateCameraCharuco(
        charucoCorners=todos_cantos,
        charucoIds=todos_ids,
        board=tabuleiro,
        imageSize=tamanho_imagem,
        cameraMatrix=None,
        distCoeffs=None
    )

    print(f"\nCalibração Concluída! Erro de reprojeção: {erro:.4f} pixels (quanto menor, melhor)")
    
    np.savez('calibracao_camera.npz', mtx=matriz_camera, dist=coef_distorcao)
    print("Dados salvos no arquivo 'calibracao_camera.npz'.")
else:
    print("Nenhuma foto foi capturada. Calibração cancelada.")

=== CALIBRAÇÃO DE CÂMERA INICIADA ===
- Pressione 'c' para CAPTURAR uma foto válida.
- Pressione 'q' quando tiver cerca de 20 a 30 fotos para FINALIZAR.
Foto 1 capturada! Mude o ângulo do tabuleiro.
Foto 2 capturada! Mude o ângulo do tabuleiro.
Foto 3 capturada! Mude o ângulo do tabuleiro.
Foto 4 capturada! Mude o ângulo do tabuleiro.
Foto 5 capturada! Mude o ângulo do tabuleiro.
Foto 6 capturada! Mude o ângulo do tabuleiro.
Foto 7 capturada! Mude o ângulo do tabuleiro.
Foto 8 capturada! Mude o ângulo do tabuleiro.
Foto 9 capturada! Mude o ângulo do tabuleiro.
Foto 10 capturada! Mude o ângulo do tabuleiro.
Foto 11 capturada! Mude o ângulo do tabuleiro.
Foto 12 capturada! Mude o ângulo do tabuleiro.
Foto 13 capturada! Mude o ângulo do tabuleiro.
Foto 14 capturada! Mude o ângulo do tabuleiro.
Foto 15 capturada! Mude o ângulo do tabuleiro.
Foto 16 capturada! Mude o ângulo do tabuleiro.
Foto 17 capturada! Mude o ângulo do tabuleiro.
Foto 18 capturada! Mude o ângulo do tabuleiro.
Foto 19 ca

In [ ]:
import cv2
import numpy as np

# --- 1. CARREGA A CALIBRAÇÃO DA CÂMERA ---
try:
    with np.load('calibracao_camera.npz') as dados:
        mtx = dados['mtx']
        dist = dados['dist']
    print("Calibração carregada com sucesso!")
except FileNotFoundError:
    print("ERRO: Arquivo 'calibracao_camera.npz' não encontrado. Rode a calibração primeiro.")
    exit()

# --- 2. CONFIGURAÇÕES DE COR OTIMIZADAS ---
AZUL_LOW = np.array([100, 110, 80]) 
AZUL_HIGH = np.array([115, 255, 255])

VERDE_LOW = np.array([75, 100, 10])
VERDE_HIGH = np.array([98, 255, 120])

# --- 3. FATOR DE EXTENSÃO (K) ---
FATOR_K = 2.6 

def encontrar_centroide(mask):
    contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contornos:
        maior_contorno = max(contornos, key=cv2.contourArea)
        if cv2.contourArea(maior_contorno) > 100:
            M = cv2.moments(maior_contorno)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                return (cx, cy)
    return None

# Inicializa a câmera
cap = cv2.VideoCapture(0)

while True:
    ret, frame_bruto = cap.read()
    if not ret: break

    # O SEGREDO DO TCC: Remove a distorção da lente em tempo real!
    frame = cv2.undistort(frame_bruto, mtx, dist, None, mtx)

    # Converte o frame corrigido para HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    mask_azul = cv2.inRange(hsv, AZUL_LOW, AZUL_HIGH)
    mask_verde = cv2.inRange(hsv, VERDE_LOW, VERDE_HIGH)

    kernel = np.ones((5,5),np.uint8)
    mask_azul = cv2.morphologyEx(mask_azul, cv2.MORPH_OPEN, kernel)
    mask_verde = cv2.morphologyEx(mask_verde, cv2.MORPH_OPEN, kernel)

    centro_azul = encontrar_centroide(mask_azul)
    centro_verde = encontrar_centroide(mask_verde)

    if centro_azul and centro_verde:
        cv2.circle(frame, centro_azul, 10, (255, 0, 0), -1)      
        cv2.circle(frame, centro_verde, 10, (0, 255, 0), -1)     

        # Calcula o vetor de direção
        vetor_x = centro_verde[0] - centro_azul[0]
        vetor_y = centro_verde[1] - centro_azul[1]

        # Projeta a ponta
        ponta_x = int(centro_verde[0] + (vetor_x * FATOR_K))
        ponta_y = int(centro_verde[1] + (vetor_y * FATOR_K))
        ponto_ponta = (ponta_x, ponta_y)

        # Desenha na tela
        cv2.line(frame, centro_azul, ponto_ponta, (255, 255, 255), 2)
        cv2.drawMarker(frame, ponto_ponta, (0, 0, 255), cv2.MARKER_CROSS, 20, 3)

    cv2.imshow('Rastreio de Caneta - Lente Calibrada', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Calibração carregada com sucesso!


In [1]:
import cv2
import numpy as np
import math
from collections import deque # IMPORTANTE: Para o buffer do rastro

# --- 1. CARREGA A CALIBRAÇÃO DA CÂMERA ---
try:
    with np.load('calibracao_camera.npz') as dados:
        mtx = dados['mtx']
        dist = dados['dist']
    print("Calibração carregada com sucesso!")
except FileNotFoundError:
    print("ERRO: Arquivo 'calibracao_camera.npz' não encontrado.")
    print("Por favor, rode o script de calibração primeiro.")
    exit()

# --- 2. CONFIGURAÇÕES DE COR OTIMIZADAS ---
# Azul (Fundo da caneta)
AZUL_LOW = np.array([100, 110, 80]) 
AZUL_HIGH = np.array([115, 255, 255])

# Verde (Meio da caneta)
VERDE_LOW = np.array([75, 100, 10])
VERDE_HIGH = np.array([98, 255, 120])

SOMBRA_LOW =np.array([0,0,0])
SOMBRA_HIGH=np.array([179,255,100])

K = 2.7 

#rastro
BUFFER_SIZE = 64 
pontos_rastro = deque(maxlen=BUFFER_SIZE)
COR_TRACO = (0, 255, 255)
ESPESSURA_TRACO = 5

def encontrar_centroide(mask):
    """Encontra o centro de massa do maior objeto na máscara."""
    contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contornos:
        maior_contorno = max(contornos, key=cv2.contourArea)
        if cv2.contourArea(maior_contorno) > 100: # filtro de ruído
            M = cv2.moments(maior_contorno)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                return (cx, cy)
    return None


cap = cv2.VideoCapture(0)

print("\n=== RASTREIO COM DESENHO INICIADO ===")
print("Pressione 'c' para limpar o desenho.")
print("Pressione 'q' para sair.")

while True:
    ret, frame_bruto = cap.read()
    if not ret: break

    # tirando distorção da lente
    frame = cv2.undistort(frame_bruto, mtx, dist, None, mtx)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Segmentação por Cor
    mask_azul = cv2.inRange(hsv, AZUL_LOW, AZUL_HIGH)
    mask_verde = cv2.inRange(hsv, VERDE_LOW, VERDE_HIGH)

    # Limpeza Morfológica
    kernel = np.ones((5,5),np.uint8)
    mask_azul = cv2.morphologyEx(mask_azul, cv2.MORPH_OPEN, kernel)
    mask_verde = cv2.morphologyEx(mask_verde, cv2.MORPH_OPEN, kernel)

    #centros
    centro_azul = encontrar_centroide(mask_azul)
    centro_verde = encontrar_centroide(mask_verde)

    if centro_azul and centro_verde:
        #desenho das marcações
        cv2.circle(frame, centro_azul, 8, (255, 0, 0), -1)      # Azul
        cv2.circle(frame, centro_verde, 8, (0, 255, 0), -1)     # Verde

        # 2. Matemática da Ponta: Vetor do Azul(fundo) -> Verde(meio)
        vetor_x = centro_verde[0] - centro_azul[0]
        vetor_y = centro_verde[1] - centro_azul[1]

        #acha a ponta a partir da marcação verde
        ponta_x = int(centro_verde[0] + (vetor_x * K))
        ponta_y = int(centro_verde[1] + (vetor_y * K))
        ponto_ponta = (ponta_x, ponta_y)
        
        #Sombra
        sombra_y_inicio = max(0, ponta_y - 20)
        sombra_y_fim = min(frame.shape[0], ponta_y + 20)
        sombra_x_inicio = max(0, ponta_x-15)
        sombra_x_fim = min(frame.shape[1], ponta_x + 25)
        cv2.rectangle(frame, (sombra_x_inicio,sombra_y_inicio), (sombra_x_fim, sombra_y_fim), (0, 255, 0), 2)
        sombra_roi = frame[sombra_y_inicio:sombra_y_fim, sombra_x_inicio:sombra_x_fim]
        sombra_hsv = cv2.cvtColor(sombra_roi, cv2.COLOR_BGR2HSV)
        sombra_mask = cv2.inRange(sombra_hsv, SOMBRA_LOW, SOMBRA_HIGH)
        sombra_centro = encontrar_centroide(sombra_mask)
        #desenho da ponta
        cv2.drawMarker(frame, ponto_ponta, (0, 0, 255), cv2.MARKER_CROSS, 15, 2)
        if sombra_centro:
            
            pontos_rastro.appendleft(ponto_ponta)
            #desenho da sombra
            cv2.circle(frame, (sombra_x_inicio+sombra_centro[0],sombra_y_inicio+sombra_centro[1]), 8, (0, 0, 255), -1)
        
    
    else:
        # Se perdeu a caneta, adiciona None para criar uma quebra no traço
        # (Isso impede que uma linha reta cruze a tela quando a caneta reaparecer)
        pontos_rastro.appendleft(None)

    # --- DESENHA O RASTRO COMPLETO (Trail) ---
    # Loop sobre o buffer de pontos
    for i in range(1, len(pontos_rastro)):
        # Se o ponto atual ou o anterior forem None, pula (quebra no traço)
        if pontos_rastro[i - 1] is None or pontos_rastro[i] is None:
            continue
        
        # Desenha a linha conectando o ponto anterior ao atual
        cv2.line(frame, pontos_rastro[i - 1], pontos_rastro[i], COR_TRACO, ESPESSURA_TRACO)

    # Exibe o resultado
    cv2.imshow('Rastreio Preciso com Rastro de Desenho', frame)

    tecla = cv2.waitKey(1) & 0xFF
    if tecla == ord('q'): # Sair
        break
    elif tecla == ord('c'): # Limpar desenho
        pontos_rastro.clear()
        print("Desenho limpo.")

cap.release()
cv2.destroyAllWindows()

Calibração carregada com sucesso!

=== RASTREIO COM DESENHO INICIADO ===
Pressione 'c' para limpar o desenho.
Pressione 'q' para sair.


error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'
